# Performance Optimization, PyArrow & Copy-on-Write (5+ Years Interview Guide)
Deep architectural guide to vectorization hierarchy, PyArrow backend in Pandas 2.0+, Copy-on-Write (CoW), Parquet binary I/O, and pd.eval engine.

### Key 5-Year Interview Concepts Covered:
- **Vectorization Hierarchy**: Vectorized NumPy/Cython > `apply()` > `itertuples()` > `iterrows()`.
- **PyArrow Backend (Pandas 2.0+)**: Zero-copy strings, native nullability, and 50-70% memory reduction with `dtype_backend='pyarrow'`.
- **Copy-on-Write (CoW)**: Eliminating `SettingWithCopyWarning` and optimizing shallow slices in Pandas 2.0/3.0 (`pd.options.mode.copy_on_write = True`).
- **High-Performance Storage (Apache Parquet)**: Columnar compression (Snappy/ZSTD) vs uncompressed CSV.
- **High-Speed Evaluator (`pd.eval` & `df.query`)**: Bypassing intermediate Python heap allocations using the NumExpr engine.

This interactive notebook is fully customized using the Fintech dataset `data/raw_transactions.csv`.

In [ ]:
# Setup imports & load dataset
import pandas as pd
import numpy as np
import os

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path, na_values=['Nan', ''])
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(2)

## Section 1: Vectorization Hierarchy & Iteration Speedups

### Iteration Benchmark: Vectorization vs `itertuples` vs `iterrows`
**Explanation**: In Python, `.iterrows()` is the slowest possible method because it creates a new `pd.Series` object for every single row in Python heap memory. `.itertuples()` yields lightweight namedtuples running ~100x faster. Native vectorized NumPy arithmetic runs in compiled C/Cython without GIL overhead, running ~1000x faster.

**Syntax**: `df['col_a'] * df['col_b']  # Vectorized C-speed`

In [ ]:
import time
sample = df[['transaction_amount']].dropna().head(5000).copy()

# 1. Vectorized calculation
t0 = time.perf_counter()
sample['fee_vec'] = sample['transaction_amount'] * 0.03
t_vec = time.perf_counter() - t0

# 2. itertuples
t0 = time.perf_counter()
fees_tuples = [row.transaction_amount * 0.03 for row in sample.itertuples()]
t_tuples = time.perf_counter() - t0

print(f'Vectorized Time: {t_vec*1000:.3f} ms')
print(f'itertuples Time: {t_tuples*1000:.3f} ms ({t_tuples/t_vec:.1f}x slower)')

### NumPy Vectorization (`np.where` & `np.select`)
**Explanation**: For conditional logic across multiple columns, avoid Python-level `df.apply(lambda row: ..., axis=1)`. Use `np.where(condition, true_val, false_val)` for binary branching and `np.select(conditions_list, choices_list, default)` for multi-branch rules. Both execute at native C speed across contiguous memory buffers.

**Syntax**: `np.where(df['amount'] > 1000, 'High', 'Low')` / `np.select(conditions, choices)`

In [ ]:
conditions = [
    df['transaction_amount'] >= 1000,
    df['transaction_amount'] >= 500,
    df['transaction_amount'] < 500
]
choices = ['Tier_1_Large', 'Tier_2_Medium', 'Tier_3_Small']
df['tier_numpy'] = np.select(conditions, choices, default='Unknown')
print(df[['transaction_amount', 'tier_numpy']].head(4))

## Section 2: PyArrow Backend & Copy-on-Write (Pandas 2.0+)

### PyArrow Backend Engine (`dtype_backend='pyarrow'`)
**Explanation**: Traditional Pandas uses Python `object` pointers for strings (consuming 8 bytes per pointer + 50+ bytes per PyObject on the heap). Pandas 2.0+ integrates Apache Arrow columnar memory layouts. Arrow strings are stored contiguously in memory with native nullability, cutting memory usage by 50-70% and accelerating string operations by 5-10x.

**Syntax**: `pd.read_csv(path, engine='pyarrow', dtype_backend='pyarrow')`

In [ ]:
try:
    df_arrow = pd.read_csv('data/raw_transactions.csv', engine='pyarrow', dtype_backend='pyarrow')
    print('Standard memory:', df.memory_usage(deep=True).sum() / 1024**2, 'MB')
    print('PyArrow memory:', df_arrow.memory_usage(deep=True).sum() / 1024**2, 'MB')
except Exception as e:
    print('PyArrow backend demo (install pyarrow to test):', e)

### Copy-on-Write Mechanics (CoW)
**Explanation**: Historically, slicing a DataFrame created ambiguous views vs copies, triggering the notorious `SettingWithCopyWarning`. With Copy-on-Write enabled (`pd.options.mode.copy_on_write = True`), Pandas guarantees that slicing a DataFrame creates a zero-copy reference that is copied lazily ONLY when mutated. This eliminates defensive copying overhead and prevents silent chained assignment bugs.

**Syntax**: `pd.options.mode.copy_on_write = True`

In [ ]:
pd.options.mode.copy_on_write = True
subset_df = df[['transaction_id', 'transaction_amount']]
# In CoW mode, modifying subset_df never alters df, and raises no warnings!
subset_df['transaction_amount'] = subset_df['transaction_amount'].fillna(0)
print('CoW slice modified safely without SettingWithCopyWarning.')

## Section 3: High-Performance Storage & `pd.eval`

### Apache Parquet Columnar Storage (`to_parquet` / `read_parquet`)
**Explanation**: In production data lakes, never use CSV for large datasets. Parquet provides columnar compression (Snappy/ZSTD), schema preservation (data types are preserved without re-parsing strings), and column pruning (reading only specific columns without loading full files from disk).

**Syntax**: `df.to_parquet('data.parquet', compression='snappy')` / `pd.read_parquet(..., columns=['id'])`

In [ ]:
os.makedirs('scratch', exist_ok=True)
df.to_parquet('scratch/transactions.parquet', compression='snappy')
read_df = pd.read_parquet('scratch/transactions.parquet', columns=['transaction_id', 'transaction_amount'])
print('Parquet loaded columns:', list(read_df.columns), 'Shape:', read_df.shape)

### High-Speed Expression Evaluation with `pd.eval()`
**Explanation**: `pd.eval()` evaluates complex multi-column algebraic expressions using the C-based NumExpr engine. Unlike standard Python which creates large temporary intermediate arrays in RAM for every operator in `(a + b) * (c - d)`, `pd.eval()` computes the entire expression in CPU L1/L2 cache chunks without heap allocation.

**Syntax**: `pd.eval('df.col_a * 1.05 + df.col_b * 0.95', engine='numexpr')`

In [ ]:
df['adjusted_amount'] = pd.eval('df.transaction_amount * 1.05 + df.account_age_months * 0.1')
print(df[['transaction_amount', 'account_age_months', 'adjusted_amount']].head(3))

## Section: Senior Fintech Interview Questions (5+ Years Experience)

### Q1: Benchmark Vectorized Fee Calculation vs Iteration
**Explanation**: Calculate a custom 2.5% transaction fee with a $1.00 minimum cap using `np.maximum()` vectorized arithmetic and compare performance against row-by-row iteration.

**Syntax**: `np.maximum(df['transaction_amount'] * 0.025, 1.0)`

In [ ]:
amounts = df['transaction_amount'].dropna()
fees_vectorized = np.maximum(amounts * 0.025, 1.0)
print('Sample Vectorized Calculated Fees:\n', fees_vectorized.head(4))